# 대회 데이터 구조 확인

원본 `data/open.zip`을 수정하거나 압축 해제하지 않고 파일 구성과 dev 데이터 구조를 확인한다.

In [ ]:
from pathlib import Path
import gzip, io, json, zipfile
import pandas as pd

ZIP_PATH = Path(r'C:/study-contests/data/open.zip')
assert ZIP_PATH.exists(), ZIP_PATH
print(ZIP_PATH, ZIP_PATH.stat().st_size)

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as z:
    infos = z.infolist()
    for info in infos:
        print(f'{info.filename}\t{info.file_size:,} bytes')
print('파일 수:', len(infos))

In [ ]:
def load_jsonl_from_zip(zip_path, member, limit=None):
    rows = []
    with zipfile.ZipFile(zip_path) as z, z.open(member) as raw:
        with gzip.GzipFile(fileobj=raw) as stream:
            for line in stream:
                if line.strip():
                    rows.append(json.loads(line.decode('utf-8')))
                    if limit and len(rows) >= limit:
                        break
    return rows

dev = load_jsonl_from_zip(ZIP_PATH, 'dev.jsonl.gz')
print('dev rows:', len(dev))
print('top keys:', sorted(dev[0]))
print('first id:', dev[0]['id'])
print('docs:', len(dev[0]['docs']), 'meta keys:', len(dev[0]['meta']))

In [ ]:
ids = [r['id'] for r in dev]
print('id 결측:', sum(not x for x in ids))
print('id 중복:', len(ids) - len(set(ids)))
print('docs 개수 분포:
', pd.Series([len(r['docs']) for r in dev]).value_counts().sort_index())
print('문서 유형:', sorted({d['type'] for r in dev for d in r['docs']}))

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as z:
    labels = pd.read_csv(io.BytesIO(z.read('dev_labels.csv')))
print(labels.shape)
print(labels.columns.tolist())
print('v 값:', sorted(set(labels.filter(regex=r'^v\d+$').to_numpy().ravel())))
print('label id 중복:', labels['id'].duplicated().sum())

## 확인 결과 기록

- 원본 ZIP은 읽기 전용으로 사용한다.
- `dev.jsonl.gz`와 `dev_labels.csv`는 `id`로 대응한다.
- 다음 단계는 `--mock` 실행과 제출 CSV 검증이다.